In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import Model
train_dir="path to your training set"
test_dir="path to your testing set"

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define ImageDataGenerator with validation split and data augmentation
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,   # Preprocess input for MobileNetV2
    validation_split=0.2,                       # 20% for validation
    horizontal_flip=True,                       # Augmentation: Flip images horizontally
    rotation_range=30,                          # Randomly rotate images
    zoom_range=0.2,                             # Zoom in and out
    width_shift_range=0.2,                      # Randomly shift images horizontally
    height_shift_range=0.2,                     # Randomly shift images vertically
    shear_range=0.2,                            # Shear transformations
    fill_mode='nearest'                         # Fill missing pixels with nearest value
)

# Training generator (80% of data, with augmentation)
train_generator = datagen.flow_from_directory(
    train_dir,                                  # Directory containing the data
    target_size=(224, 224),                     # Resize images to match model input size
    batch_size=32,                              # Batch size for training
    class_mode='binary',                        # Binary classification (malignant or benign)
    subset='training'                           # Specify this is for training
)

# Validation generator (20% of data, no augmentation)
val_generator = datagen.flow_from_directory(
    train_dir,                                  # Directory containing the data
    target_size=(224, 224),                     # Resize images to match model input size
    batch_size=32,                              # Batch size for validation
    class_mode='binary',                        # Binary classification (malignant or benign)
    subset='validation'                         # Specify this is for validation
)

In [ ]:
#Building the model
from tensorflow.keras import layers, models
Base_Model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x=Base_Model.output
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(512,activation='relu')(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(256,activation='relu')(x)
x=layers.Dropout(0.2)(x)

prediction_layer=layers.Dense(1,activation='sigmoid')(x)

model=Model(inputs=Base_Model.input,outputs=prediction_layer)
print(model.summary())

In [ ]:
for layer in model.layers[:-5]:
  layer.trainable=False
  from tensorflow.keras.optimizers import Adam
epochs=6
optimizer= Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])

In [ ]:
model.fit(train_generator,epochs=epochs,validation_data=val_generator)
path_for_saved_model="add path"
model.save(path_for_saved_model)

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load the saved model
model = load_model('model path')

# Preprocess the test set
test_dir = "test directory path"

test_datagen = ImageDataGenerator(rescale=1.0/255)  # Normalize images
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),  # Same size as used during training
    batch_size=64,
    class_mode='binary',  # Adjust if multi-class: 'categorical'
    shuffle=False
)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(test_generator)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")


Found 2000 images belonging to 2 classes.
32/32 ━━━━━━━━━━━━━━━━━━━━ 35s 929ms/step - accuracy: 0.9124 - loss: 0.2133
Test Loss: 0.5008369088172913
Test Accuracy: 0.7695000171661377


In [8]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array 
import numpy as np
import os

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load the saved model
model = load_model('C:/Users/prajw/OneDrive/Documents/Desktop/MINI PROJECT ONCONOVA/mymodel.h5')

# Load and preprocess the image
def preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))  # Resize to model's input size
    img_array = img_to_array(img)  # Convert to array
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    img_array = img_array / 255.0  # Normalize to [0, 1]
    return img_array

# Path to the image(s) you want to test
image_path = "C:/Users/prajw/Downloads/ben2.png" # Replace with the path to your image

# Verify that the image exists
if os.path.exists(image_path):
    # Preprocess the image
    img = preprocess_image(image_path)

    # Predict the class
    prediction = model.predict(img)[0][0]  # Since it's binary classification
    label = "malignant" if prediction > 0.5 else "benign"

    print(f"Image: {image_path}, Predicted Class: {label}, Confidence: {prediction}")
else:
    print(f"Error: The image at {image_path} does not exist.")



1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Image: C:/Users/prajw/Downloads/ben2.png, Predicted Class: benign, Confidence: 0.1651744693517685
